In [132]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

In [133]:
texts = [
    "I love this movie",
    "This film is amazing",
    "Very good acting",
    "Excellent story",
    "I hate this movie",
    "Terrible film",
    "Very boring story",
    "Worst acting ever"
    "This movie was awful"
]
 
labels = np.array([1, 1, 1, 1, 0, 0, 0, 0])

In [134]:
vocab_size = 1000
max_length = 8

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)

X = pad_sequences(sequences, maxlen=max_length, padding='post')

print("Word Index:")
print(tokenizer.word_index)

print("\nInput Sequences:")
print(X)

Word Index:
{'<OOV>': 1, 'this': 2, 'movie': 3, 'i': 4, 'film': 5, 'very': 6, 'acting': 7, 'story': 8, 'love': 9, 'is': 10, 'amazing': 11, 'good': 12, 'excellent': 13, 'hate': 14, 'terrible': 15, 'boring': 16, 'worst': 17, 'everthis': 18, 'was': 19, 'awful': 20}

Input Sequences:
[[ 4  9  2  3  0  0  0  0]
 [ 2  5 10 11  0  0  0  0]
 [ 6 12  7  0  0  0  0  0]
 [13  8  0  0  0  0  0  0]
 [ 4 14  2  3  0  0  0  0]
 [15  5  0  0  0  0  0  0]
 [ 6 16  8  0  0  0  0  0]
 [17  7 18  3 19 20  0  0]]


In [135]:
class TokenAndPositionEmbedding(layers.Layer): #1
    def __init__(self, max_length, vocab_size, embed_dim):
        super().__init__()
        self.token_embedding = layers.Embedding(
            input_dim=vocab_size,
            output_dim=embed_dim
        )
 
        self.position_embedding = layers.Embedding(
            input_dim=max_length,
            output_dim=embed_dim
        )
 
    def call(self, x):
        positions = tf.range(start=0, limit=tf.shape(x)[-1], delta=1)
        token_emb = self.token_embedding(x)
        position_emb = self.position_embedding(positions)
        return token_emb + position_emb

In [136]:
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )
 
        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim)
        ])
 
        self.layernorm1 = layers.LayerNormalization()
        self.layernorm2 = layers.LayerNormalization()
 
    def call(self, inputs):
        # Self-attention (Query, Key, Value are calculated here using the same input)
        # The attention layer takes the input and computes
        # attention scores to capture relationships between different positions in the sequence.
        attention_output = self.attention(inputs, inputs)
 
        # Add + Normalize
        out1 = self.layernorm1(inputs + attention_output)
 
        # Feed-forward network
        ffn_output = self.ffn(out1)
 
        # Add + Normalize
        out2 = self.layernorm2(out1 + ffn_output)
 
        return out2

In [137]:
# build the model
embed_dim = 64
num_heads = 8
ff_dim = 32
inputs = layers.Input(shape=(max_length,))
 
x = TokenAndPositionEmbedding(
    max_length=max_length,
    vocab_size=vocab_size,
    embed_dim=embed_dim
)(inputs)
 
x = TransformerBlock(
    embed_dim=embed_dim,
    num_heads=num_heads,
    ff_dim=ff_dim
)(x)
 
x = layers.GlobalAveragePooling1D()(x)
 
outputs = layers.Dense(1, activation="sigmoid")(x)
 
model = tf.keras.Model(inputs=inputs, outputs=outputs)

In [138]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

model.summary() 

Model: "functional_27"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_30 (InputLayer)     │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding_15 │ (None, 8, 64)          │        64,512 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_14            │ (None, 8, 64)          │       137,120 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_14     │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_43 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 201,697 (787.88 KB)

 Trainable params: 201,697 (787.88 KB)

 Non-trainable params: 0 (0.00 B)

In [139]:
model.fit(X, labels, epochs=30, batch_size=3, verbose=1)

Epoch 1/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - accuracy: 0.3750 - loss: 1.5982
Epoch 2/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.3750 - loss: 0.7768
Epoch 3/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.5000 - loss: 0.8541
Epoch 4/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.6250 - loss: 0.6553
Epoch 5/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.6250 - loss: 0.6115
Epoch 6/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.6250 - loss: 0.6951
Epoch 7/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6250 - loss: 0.6035
Epoch 8/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.7500 - loss: 0.5091
Epoch 9/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.7500 - loss: 0.5411
Epoch 10/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.7500 - loss: 0.4882
Epoch 11/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.8750 - loss: 0.4181
Epoch 12/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7500 - loss: 0.4095
E

In [140]:
test_sentences = [
    "I love the film",
    "This movie was awful",
    "I had an amazing experience",
    "This movie was terrible"
]
 
test_seq = tokenizer.texts_to_sequences(test_sentences)
test_pad = pad_sequences(test_seq, maxlen=max_length, padding="post")
predictions = model.predict(test_pad)
 
for sentence, prediction in zip(test_sentences, predictions):
    print(sentence, "->", prediction[0])
    if prediction[0] > 0.5:
        print("Prediction: Positive")
    else:
        print("Prediction: Negative")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 273ms/step
I love the film -> 0.9501277
Prediction: Positive
This movie was awful -> 0.010345242
Prediction: Negative
I had an amazing experience -> 0.75206566
Prediction: Positive
This movie was terrible -> 0.0029444445
Prediction: Negative


Modifications to Try

embed_dim=16
num_heads=2
ff_dim=32
max_length =8

Sentence
   ↓
Token Embedding
   ↓
Positional Embedding
   ↓
Multi-Head Attention
   ↓
Feed Forward Network
   ↓
Add & Normalize
   ↓
Output Layer
   ↓
Prediction

Token Embedding:
    Token Embedding converts each word into a meaningful vector (list of numbers).
    Neural Networks won't be able to perform the math on raw text, they need continuous numbers.
    The model crashes and not be able to process the input sequences.

Position Embedding:
    It creates a unique vector representing the physical position of a word in the sentence and adds it to the token embedding.
    RNN's process single words at a time,while the Transformers process all words simultaneously.the model has absolutely no concept of word order.
    The model would treat the sentence as a random "bag of words."

Multi-Head Attention:
    It compares every word in the sentence to every other word, assigning "attention scores" to figure out which words are most relevant to each other.
    It gives the model context.So it is needed.
    If removed it fails to understand the overall context of the sentence.

Feed Forward Network:
    neural network applied to each position individually
    It is required to understand the information gathered by attention
    The model would struggle to learn complex rules and its predictive accuracy would drop significantly.

Add & Normalize:
    Add takes the input of a layer and adds it to the output,Normalize stabilizes the numbers so they don't get too large or too small.
    It is required to prevents the model from forgetting earlier information.
    If removed,it would become incredibly unstable and might completely fail to train.





PART 4

embed_dim = 16
num_heads = 2
ff_dim = 32

Time Required:10 sec
Accuracy: 


embed_dim = 8
num_heads = 1
ff_dim = 32

Time Required: 9.8 sec

Accuracy: 1 epoch = 


embed_dim = 32
num_heads = 4
ff_dim = 32

Time Required: 9.8sec
Accuracy: 1 epoch = 


embed_dim = 64
num_heads = 8
ff_dim = 32

Time Required: 6.7sec
Accuracy: 1 epoch = 14